### Calculate Metrics (KGE, NSE, PBIAS)

For multiple experimental runs, you can process each input CSV and generate corresponding output CSVs with the Issue flag:

In [4]:
# -*- coding: utf-8 -*-
"""
Hydrology Model Evaluation Script
- Multiple station filters
- Consistent stations across plots (intersection across runs)
- Vectorized metrics
- YEAR+JDAY date filtering
- Broken-axis CDF
- Spatial plots with basin overlay
"""

import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Set
from itertools import cycle
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from pathlib import Path
from typing import Optional, Set, List
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# METRIC DESCRIPTIONS
# =============================================================================
METRIC_DESCRIPTIONS: Dict[str, Dict[str, str]] = {
    'KGE': {'Name': 'Kling-Gupta Efficiency', 'Description': 'Combines correlation, bias, variability.',
            'Range': '(-∞,1]', 'Units': 'Dimensionless', 'Interpretation': 'Higher better'},
    'NKGE': {'Name': 'Normalized KGE', 'Description': 'Normalized form of KGE', 'Range': '(0,1]',
             'Units': 'Dimensionless', 'Interpretation': 'Higher better'},
    'NSE': {'Name': 'Nash-Sutcliffe Efficiency', 'Description': 'Variance explained', 'Range': '(-∞,1]',
            'Units': 'Dimensionless', 'Interpretation': 'Higher better'},
    'PBIAS': {'Name': 'Percent Bias', 'Description': 'Mean bias', 'Range': '(-∞,∞)', 'Units': '%',
              'Interpretation': 'Closer to 0 better'},
    'RMSE': {'Name': 'Root Mean Square Error', 'Description': 'Error magnitude', 'Range': '[0,∞)',
             'Units': 'units', 'Interpretation': 'Lower better'},
    'MAE': {'Name': 'Mean Absolute Error', 'Description': 'Absolute error', 'Range': '[0,∞)',
            'Units': 'units', 'Interpretation': 'Lower better'},
    'R2': {'Name': 'Coefficient of Determination', 'Description': 'Variance explained', 'Range': '[0,1]',
           'Units': 'Dimensionless', 'Interpretation': 'Higher better'},
    'MAPE': {'Name': 'Mean Absolute Percentage Error', 'Description': 'Relative error', 'Range': '[0,∞)',
             'Units': '%', 'Interpretation': 'Lower better'},
    'VE': {'Name': 'Volume Error', 'Description': 'Volumetric bias', 'Range': '(-∞,∞)',
           'Units': '%', 'Interpretation': 'Closer to 0 better'}
}

# =============================================================================
# METRIC CALCULATION
# =============================================================================
def compute_metrics_vectorized(sim: pd.Series, obs: pd.Series):
    df = pd.DataFrame({'sim': sim, 'obs': obs}).dropna()
    if len(df) < 2:
        return {m: np.nan for m in METRIC_DESCRIPTIONS}, "Insufficient data"
    sim, obs = df['sim'], df['obs']
    metrics, issue = {}, None
    mean_sim, mean_obs = sim.mean(), obs.mean()
    std_sim, std_obs = sim.std(ddof=1), obs.std(ddof=1)
    sum_obs = obs.sum()

    # KGE + NKGE
    if mean_obs == 0 or std_obs == 0 or std_sim == 0:
        metrics['KGE'] = metrics['NKGE'] = np.nan
        issue = "Zero mean/std in KGE"
    else:
        r = np.corrcoef(sim, obs)[0, 1]
        if np.isnan(r):
            metrics['KGE'] = metrics['NKGE'] = np.nan
            issue = "Invalid correlation"
        else:
            alpha = std_sim / std_obs
            beta = mean_sim / mean_obs
            kge = 1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2)
            metrics['KGE'] = kge
            metrics['NKGE'] = 1 / (2 - kge)

    # NSE
    if std_obs == 0:
        metrics['NSE'] = np.nan
        issue = issue or "Zero std in NSE"
    else:
        metrics['NSE'] = 1 - ((obs - sim) ** 2).sum() / ((obs - mean_obs) ** 2).sum()

    # PBIAS + VE
    if sum_obs == 0:
        metrics['PBIAS'] = metrics['VE'] = np.nan
        issue = issue or "Zero sum obs"
    else:
        bias = (sim.sum() - sum_obs) * 100 / sum_obs
        metrics['PBIAS'] = metrics['VE'] = bias

    metrics['RMSE'] = np.sqrt(((obs - sim) ** 2).mean())
    metrics['MAE'] = (obs - sim).abs().mean()

    # R2
    if std_obs == 0 or std_sim == 0:
        metrics['R2'] = np.nan
        issue = issue or "Zero std in R2"
    else:
        r = np.corrcoef(sim, obs)[0, 1]
        metrics['R2'] = r ** 2 if not np.isnan(r) else np.nan

    # MAPE
    nonzero = obs != 0
    if nonzero.sum() < 2:
        metrics['MAPE'] = np.nan
        issue = issue or "Insufficient non-zero obs"
    else:
        metrics['MAPE'] = ((obs[nonzero] - sim[nonzero]).abs() / obs[nonzero].abs()).mean() * 100

    return metrics, issue

# =============================================================================
# PROCESSING
# =============================================================================
def compute_efficiency_custom(df: pd.DataFrame, prefix_obs='QOMEAS_', prefix_simu='QOSIM_') -> pd.DataFrame:
    results = []
    for col_obs in [c for c in df.columns if c.startswith(prefix_obs)]:
        sid = col_obs[len(prefix_obs):].strip()
        col_sim = f"{prefix_simu}{sid}"
        if col_sim not in df.columns:
            results.append({'StationID': sid, **{m: np.nan for m in METRIC_DESCRIPTIONS},
                            'Issue': True, 'IssueMessage': 'Missing simulated column'})
            continue
        metrics, issue = compute_metrics_vectorized(df[col_sim], df[col_obs])
        row = {'StationID': str(sid), **metrics, 'Issue': issue is not None, 'IssueMessage': issue or ''}
        results.append(row)
    return pd.DataFrame(results).sort_values('StationID').reset_index(drop=True)

def save_metric_descriptions(outdir: Path, filename='metrics_description.txt'):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    p = outdir / filename
    with p.open('w', encoding='utf-8') as f:
        f.write("Hydrology Metrics Description\n" + "=" * 30 + "\n\n")
        for m, info in METRIC_DESCRIPTIONS.items():
            f.write(f"{info['Name']} ({m})\n")
            for k in ['Description', 'Range', 'Units', 'Interpretation']:
                f.write(f"{k}: {info[k]}\n")
            f.write("-" * 30 + "\n\n")

def process_flow_csv(input_csv: Path, output_csv: Path, prefix_obs='QOMEAS_', prefix_simu='QOSIM_',
                     skip_days=0, missing_value=None, date_column="Date",
                     start_date: Optional[str] = None, end_date: Optional[str] = None):
    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    if not input_csv.exists():
        raise FileNotFoundError(input_csv)
    df = pd.read_csv(input_csv)
    if missing_value is not None:
        df = df.replace(missing_value, np.nan)

    # YEAR + JDAY date filtering
    if "YEAR" in df.columns and "JDAY" in df.columns:
        try:
            df["Date"] = pd.to_datetime(df["YEAR"].astype(str), format="%Y") + pd.to_timedelta(df["JDAY"] - 1, unit="D")
            if start_date:
                df = df[df["Date"] >= pd.to_datetime(start_date)]
            if end_date:
                df = df[df["Date"] <= pd.to_datetime(end_date)]
            df = df.reset_index(drop=True)
        except Exception as e:
            print(f"Warning: YEAR/JDAY parse failed: {e}")
    else:
        if date_column in df.columns:
            df[date_column] = pd.to_datetime(df[date_column], errors='coerce')
            if start_date:
                df = df[df[date_column] >= pd.to_datetime(start_date)]
            if end_date:
                df = df[df[date_column] <= pd.to_datetime(end_date)]
            df = df.reset_index(drop=True)
        else:
            print("Warning: No YEAR/JDAY or Date column found; skipping date filtering.")

    if skip_days > 0:
        if skip_days >= len(df):
            raise ValueError("skip_days exceeds rows")
        df = df.iloc[skip_days:].reset_index(drop=True)

    out = compute_efficiency_custom(df, prefix_obs, prefix_simu)
    out['StationID'] = out['StationID'].astype(str)
    out.to_csv(output_csv, index=False)
    save_metric_descriptions(output_csv.parent)
    if out['Issue'].sum():
        print(f"{out['Issue'].sum()} stations had issues.")

def process_all_runs(base_path: Path, mesh_versions: List[str], gru_types: List[str], skip_days=10,
                     missing_value=None, date_column="Date",
                     start_date: Optional[str] = None, end_date: Optional[str] = None) -> List[Path]:
    base_path = Path(base_path)
    outs: List[Path] = []
    for mesh in mesh_versions:
        for gru in gru_types:
            p = base_path / mesh / gru
            if not p.is_dir():
                continue
            for sub in p.iterdir():
                if not sub.is_dir():
                    continue
                inp = sub / "MESH_output_streamflow.csv"
                out = sub / f"metrics_{mesh}_{gru}_{sub.name}.csv"
                if not inp.exists():
                    continue
                process_flow_csv(inp, out, skip_days=skip_days, missing_value=missing_value,
                                 date_column=date_column, start_date=start_date, end_date=end_date)
                outs.append(out)
    return outs

# =============================================================================
# STATION FILTERING
# =============================================================================
FilterCondition = Tuple[str, str]

def apply_station_filters(gdf: gpd.GeoDataFrame, filters: Optional[List[FilterCondition]]):
    if not filters:
        return gdf.copy()
    out = gdf.copy()
    for col, cond in filters:
        if col not in out.columns:
            raise ValueError(f"{col} missing")
        col_esc = f"`{col}`" if not re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', col) else col
        try:
            before = len(out)
            out = out.query(f"{col_esc} {cond}")
            print(f"Filter {col} {cond}: {before}→{len(out)}")
        except Exception as e:
            raise ValueError(f"Invalid filter {col} {cond}: {e}")
    return out

def load_stations_with_filters(stations_file: Path, station_id_col: str, filters: Optional[List[FilterCondition]] = None) -> pd.DataFrame:
    stations_file = Path(stations_file)
    gdf = gpd.read_file(stations_file)
    if gdf.crs and gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs(4326)
    gdf = apply_station_filters(gdf, filters or [])
    return pd.DataFrame({
        'StationID': gdf[station_id_col].astype(str),
        'Longitude': gdf.geometry.x,
        'Latitude': gdf.geometry.y
    })

# =============================================================================
# METRICS CACHE
# =============================================================================
class MetricsCache:
    def __init__(self, csv_files: List[Path]):
        self.csv_files: List[Path] = [Path(p) for p in csv_files]
        self.dfs: List[pd.DataFrame] = []
        self.run_names: List[str] = []
        self._load_all()

    def _load_all(self):
        print(f"Loading {len(self.csv_files)} metric files...")
        for csv in self.csv_files:
            if not Path(csv).exists():
                print(f"Missing: {csv}")
                continue
            df = pd.read_csv(csv)
            if 'StationID' not in df.columns:
                print(f"Skipping invalid: {Path(csv).name}")
                continue
            df['StationID'] = df['StationID'].astype(str)
            name = Path(csv).stem.replace('metrics_', '').replace('_', ' ').title()
            self.run_names.append(name)
            self.dfs.append(df)
        print(f"Loaded {len(self.dfs)} valid runs.")

    def get_metric_data(self, metric: str, allowed_stations: Optional[Set[str]] = None):
        if metric not in METRIC_DESCRIPTIONS:
            raise ValueError(f"Invalid metric: {metric}")
        if not self.dfs:
            raise ValueError("No metric data loaded.")

        allowed_set: Optional[Set[str]] = set(allowed_stations) if allowed_stations is not None else None

        valid_sets: List[Set[str]] = []
        filtered_dfs: List[pd.DataFrame] = []

        for csv, df in zip(self.csv_files, self.dfs):
            if metric not in df.columns:
                raise ValueError(f"Metric '{metric}' missing in {Path(csv).name}")
            mask = df['StationID'].isin(allowed_set) if allowed_set is not None else slice(None)
            f = df.loc[mask].copy()
            f['StationID'] = f['StationID'].astype(str)
            valid = set(f[f[metric].notna()]['StationID'])
            valid_sets.append(valid)
            filtered_dfs.append(f)

        common: Set[str] = set.intersection(*valid_sets) if valid_sets else set()
        if allowed_set is not None:
            common &= allowed_set

        all_stations = set().union(*[set(df['StationID'].astype(str)) for df in self.dfs]) if self.dfs else set()
        excluded = all_stations - common
        if excluded:
            excl_file = Path(self.csv_files[0]).parent / f"excluded_stations_{metric.lower()}.txt"
            excl_file.write_text("\n".join(sorted(excluded)), encoding='utf-8')
            print(f"Excluded {len(excluded)} stations → {excl_file.name}")

        # Also persist the included (common) stations for transparency
        if self.csv_files:
            inc_file = Path(self.csv_files[0]).parent / f"common_stations_{metric.lower()}.txt"
            inc_file.write_text("\n".join(sorted(common)), encoding='utf-8')
            print(f"Common {len(common)} stations → {inc_file.name}")

        return filtered_dfs, common, self.run_names

    def common_stations(self, metric: str, allowed_stations: Optional[Set[str]] = None) -> Set[str]:
        """Convenience: intersection across runs for a metric, intersected with allowed stations if given."""
        _, common, _ = self.get_metric_data(metric, allowed_stations=allowed_stations)
        return common

# =============================================================================
# PLOTTING UTILITIES
# =============================================================================
def _metric_unit(metric: str) -> str:
    if metric in ["PBIAS", "MAPE", "VE"]:
        return "(%)"
    if metric in ["RMSE", "MAE"]:
        return "(units)"
    return ""

# =============================================================================
# CDF PLOT
# =============================================================================
def plot_cumulative_distribution(
    cache: MetricsCache,
    metric='KGE',
    output_file='cdf.png',
    common_stations: Optional[Set[str]] = None,
    legend_title: str = "Runs",
    legend_labels: Optional[List[str]] = None
):
    output_file = Path(output_file)
    dfs, common, names = cache.get_metric_data(metric, allowed_stations=common_stations)

    if legend_labels is not None and len(legend_labels) == len(names):
        names = legend_labels  # override default names

    if not common:
        print("No common stations across runs for this metric. Skipping CDF.")
        return

    plt.figure(figsize=(10 + len(dfs)//3, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    styles = cycle(['-', '--', '-.', ':'])
    all_vals = []

    for i, (df, name) in enumerate(zip(dfs, names)):
        data = df[df['StationID'].isin(common)][metric].dropna()
        if data.empty:
            continue
        x = np.sort(data)
        y = np.linspace(0, 1, len(x))
        plt.plot(x, y, label=name, color=colors[i % 10], linestyle=next(styles), linewidth=2)
        all_vals.extend(x.tolist())

    if all_vals:
        mn, mx = min(all_vals), max(all_vals)
        r = mx - mn
        m = 0.02 * r if r > 0 else 0.02
        plt.xlim(mn - m, mx + m)

    plt.title(f"CDF of {metric} ({len(common)} Stations)")
    plt.xlabel(f"{metric} {_metric_unit(metric)}")
    plt.ylabel("Cumulative Probability")
    plt.grid(True, ls='--', alpha=0.7)
    plt.legend(title=legend_title, loc="upper left")  # 👈 user-defined
    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"CDF saved: {output_file.name}")

# =============================================================================
# BROKEN-AXIS CDF
# =============================================================================
def plot_broken_axis_cumulative_distribution(
    cache: MetricsCache,
    metric='KGE',
    output_file='cdf_broken.png',
    common_stations: Optional[Set[str]] = None,
    zoom_range: Optional[Tuple[float, float]] = None,
    legend_title: str = "Runs",
    legend_labels: Optional[List[str]] = None
):
    output_file = Path(output_file)
    dfs, common, names = cache.get_metric_data(metric, allowed_stations=common_stations)

    if legend_labels is not None and len(legend_labels) == len(names):
        names = legend_labels

    if not common:
        print("No common stations across runs for this metric. Skipping broken-axis CDF.")
        return

    curves = []
    all_vals = []
    for df, name in zip(dfs, names):
        data = df[df['StationID'].isin(common)][metric].dropna()
        if data.empty:
            continue
        x = np.sort(data)
        y = np.linspace(0, 1, len(x))
        curves.append((x, y, name))
        all_vals.extend(x.tolist())

    if not all_vals:
        print("No data for broken-axis CDF.")
        return

    mn, mx = min(all_vals), max(all_vals)
    r = mx - mn
    m = 0.02 * r if r > 0 else 0.02
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    styles = cycle(['-', '--', '-.', ':'])

    if zoom_range is None:
        plt.figure(figsize=(10 + len(curves)//3, 6))
        for i, (x, y, name) in enumerate(curves):
            plt.plot(x, y, label=name, color=colors[i % 10], linestyle=next(styles), linewidth=2)
        plt.xlim(mn - m, mx + m)
        plt.title(f"CDF of {metric} ({len(common)} Stations)")
        plt.xlabel(f"{metric} {_metric_unit(metric)}")
        plt.ylabel("Cumulative Probability")
        plt.grid(True, ls='--', alpha=0.7)
        plt.legend(title=legend_title, loc="upper left")  # 👈 here too
        plt.tight_layout()
        plt.savefig(output_file, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"CDF saved: {output_file.name}")
        return

    # With zoom_range: broken axis
    z0_raw, z1_raw = zoom_range
    z0 = max(z0_raw, mn)
    z1 = min(z1_raw, mx)
    if z0 >= z1:
        print("Invalid zoom range; using normal CDF.")
        return plot_cumulative_distribution(cache, metric, output_file, common_stations)

    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True,
                                   figsize=(12 + len(curves)//3, 6),
                                   gridspec_kw={'width_ratios': [1, 1]})

    for i, (x, y, name) in enumerate(curves):
        c = colors[i % 10]; s = next(styles)
        ax1.plot(x, y, color=c, linestyle=s, linewidth=2, label=name)
        ax2.plot(x, y, color=c, linestyle=s, linewidth=2)

    ax1.set_xlim(mn - m, z0)
    ax2.set_xlim(z0, z1)
    ax1.spines['right'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.tick_params(labelleft=False)

    d = .015
    kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
    ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)
    ax1.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
    kwargs.update(transform=ax2.transAxes)
    ax2.plot((-d, +d), (-d, +d), **kwargs)
    ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)

    ax1.legend(title=legend_title, loc="upper left")  # 👈 user-defined

    plt.suptitle(f"CDF of {metric} ({len(common)} Stations)")
    for ax in (ax1, ax2):
        ax.set_xlabel(f"{metric} {_metric_unit(metric)}")
    ax1.set_ylabel("Cumulative Probability")

    for ax in plt.gcf().axes:
        ax.grid(True, ls='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"CDF saved: {output_file.name}")
    
def _normalize_basin_path(p: Path) -> Path:
    p = Path(p)
    return p if p.suffix.lower() == ".shp" else p.with_suffix(".shp")

def plot_spatial_stations_metrics(
    cache,
    coords: pd.DataFrame,
    metric='KGE',
    run_index=0,
    run_labels: Optional[List[str]] = None,
    output_file='spatial.png',
    common_stations: Optional[Set[str]] = None,
    basin_shapefile: Optional[Path] = None,
    point_size=20,
    basin_linewidth=0.15,
    alpha=0.9  # ✅ optional transparency
):
    output_file = Path(output_file)

    if not cache.dfs:
        print("No runs loaded. Skipping spatial plot.")
        return

    run_index = min(max(run_index, 0), len(cache.dfs) - 1)

    # Label selection
    if run_labels and run_index < len(run_labels):
        name = run_labels[run_index]
    else:
        name = cache.run_names[run_index]

    df = cache.dfs[run_index]

    # Enforce consistent intersection
    _, common, _ = cache.get_metric_data(metric, allowed_stations=common_stations)
    if not common:
        print(f"No common stations available for spatial plot of {name}.")
        return

    df = df[['StationID', metric]].copy()
    df['StationID'] = df['StationID'].astype(str)

    coords = coords.copy()
    coords['StationID'] = coords['StationID'].astype(str)

    plot_df = coords[coords['StationID'].isin(common)].merge(df, on='StationID').dropna(subset=[metric])

    if plot_df.empty:
        print(f"No spatial data for {name}")
        return

    plt.figure(figsize=(10, 8))
    ax = plt.gca()

    if basin_shapefile:
        bp = _normalize_basin_path(basin_shapefile)
        if bp.exists():
            try:
                basin = gpd.read_file(bp)
                if basin.crs and basin.crs.to_string() != "EPSG:4326":
                    basin = basin.to_crs(4326)
                basin.boundary.plot(ax=ax, color='black', linewidth=basin_linewidth)
                basin.plot(ax=ax, facecolor='none', edgecolor='gray',
                           linewidth=basin_linewidth * 0.6, alpha=0.3)
            except Exception as e:
                print(f"Warning: basin load failed: {e}")

    vmin, vmax = plot_df[metric].min(), plot_df[metric].max()
    r = vmax - vmin
    m = 0.02 * r if r > 0 else 0.02

    sc = plt.scatter(
        plot_df['Longitude'],
        plot_df['Latitude'],
        c=plot_df[metric],
        cmap='viridis',
        s=point_size,
        edgecolors='none',   # ✅ removed borders
        alpha=alpha,
        vmin=vmin - m,
        vmax=vmax + m
    )

    plt.colorbar(sc, label=f"{metric}")
    plt.title(f"{metric}: {name} ({len(plot_df)} stations)")
    plt.xlabel("Longitude (°)")
    plt.ylabel("Latitude (°)")
    plt.grid(True, ls='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()

    print(f"Spatial plot saved: {output_file.name}")

def plot_spatial_station_metrics_comparison(
    cache,
    coords: pd.DataFrame,
    metric='KGE',
    run_index1=0,
    run_index2=1,
    run_labels: Optional[List[str]] = None,
    output_file='spatial_cmp.png',
    common_stations: Optional[Set[str]] = None,
    basin_shapefile: Optional[Path] = None,
    point_size=20,
    basin_linewidth=0.15,
    max_name_length: int = 20,
    alpha=0.9  # ✅ optional transparency
):

    def shorten(text, max_len):
        return text if len(text) <= max_len else text[:max_len - 3] + "..."

    output_file = Path(output_file)

    if len(cache.dfs) < 2:
        print("Need ≥2 runs for comparison.")
        return

    i1 = min(max(run_index1, 0), len(cache.dfs) - 1)
    i2 = min(max(run_index2, 0), len(cache.dfs) - 1)

    # Label selection
    if run_labels:
        name1 = run_labels[i1] if i1 < len(run_labels) else cache.run_names[i1]
        name2 = run_labels[i2] if i2 < len(run_labels) else cache.run_names[i2]
    else:
        name1, name2 = cache.run_names[i1], cache.run_names[i2]

    name1 = shorten(name1, max_name_length)
    name2 = shorten(name2, max_name_length)

    # Enforce intersection
    _, common, _ = cache.get_metric_data(metric, allowed_stations=common_stations)
    if not common:
        print("No common stations for comparison plot.")
        return

    df1 = cache.dfs[i1][['StationID', metric]].copy().rename(columns={metric: f"{metric}_1"})
    df2 = cache.dfs[i2][['StationID', metric]].copy().rename(columns={metric: f"{metric}_2"})

    df1['StationID'] = df1['StationID'].astype(str)
    df2['StationID'] = df2['StationID'].astype(str)

    coords = coords.copy()
    coords['StationID'] = coords['StationID'].astype(str)

    merged = df1.merge(df2, on='StationID')
    merged = merged[merged['StationID'].isin(common)]
    merged['Diff'] = merged[f"{metric}_1"] - merged[f"{metric}_2"]

    plot_df = coords.merge(merged, on='StationID').dropna(subset=['Diff'])

    if plot_df.empty:
        print("No comparison data.")
        return

    plt.figure(figsize=(10, 8))
    ax = plt.gca()

    if basin_shapefile:
        bp = _normalize_basin_path(basin_shapefile)
        if bp.exists():
            try:
                basin = gpd.read_file(bp)
                if basin.crs and basin.crs.to_string() != "EPSG:4326":
                    basin = basin.to_crs(4326)
                basin.boundary.plot(ax=ax, color='black', linewidth=basin_linewidth)
                basin.plot(ax=ax, facecolor='none', edgecolor='black',
                           linewidth=basin_linewidth * 0.6, alpha=0.3)
            except Exception as e:
                print(f"Warning: basin load failed: {e}")

    diff = plot_df['Diff']
    bound = max(diff.abs().max(), 0.1)

    sc = plt.scatter(
        plot_df['Longitude'],
        plot_df['Latitude'],
        c=diff,
        cmap='RdBu',
        s=point_size,
        edgecolors='none',   # ✅ removed borders
        alpha=alpha,
        vmin=-bound,
        vmax=bound
    )

    plt.colorbar(sc, label=f"{metric} Difference")
    plt.title(f"{metric} Difference: {name1} vs {name2} ({len(plot_df)} stations)")
    plt.xlabel("Longitude (°)")
    plt.ylabel("Latitude (°)")
    plt.grid(True, ls='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.close()

    print(f"Comparison plot saved: {output_file.name}")

In [5]:
# =============================================================================
# USAGE #
# =============================================================================
if __name__ == "__main__":
    # Paths & settings
    base_path = Path(r"D:\Zelalem\RUNs")
    station_id_col = "Obs_NM"
    stations_file = Path(r"D:\Zelalem\WSC\combined_discharge_stations_comids.gpkg")
    basin_file = Path(r"D:\Zelalem\WSC\CanTrans_MERIT_StudyDomain.shp")

    date_column = "Date"
    start_date = "1980-10-01"
    end_date = "2024-09-30"

    # mesh_versions = ["MESH_CaSRv2p1", "MESH_CaSRv3p1", "MESH_CaSRv3p2", "MESH_ERA5L"]
    mesh_versions = ["MESH_CaSRv3p2"]
    gru_types = ["Average_GRU_Params"]
    #gru_types = ["RTE", "IRFroutedRunoff", "KWTroutedRunoff"]
    skip_days = 365
    missing_value = -1

    # 1) Load station coordinates (optionally with filters) from your GPKG
    coords = load_stations_with_filters(
        stations_file,
        station_id_col=station_id_col,
        filters=[
            ("PRecord", ">= 10"),
#            ("Sub_Reg", "== 'QC'")
#            ("PRecord", ">= 10"),
#            ("DA_Diff", "> -10"),
#            ("DA_Diff", "< 10"),
            ("HYD_STATUS", "== 'A'")
        ]
    )

    # 2) Build the metrics CSVs for all runs and load cache
    print("Processing all model runs...")
    output_csvs = process_all_runs(
        base_path, mesh_versions, gru_types,
        skip_days=skip_days, missing_value=missing_value,
        date_column=date_column, start_date=start_date, end_date=end_date
    )
    if not output_csvs:
        raise RuntimeError("No output CSVs generated.")

    print("\nLoading and filtering stations...")
    cache = MetricsCache(output_csvs)

    # 3) Decide the station set ONCE and reuse everywhere (intersection across runs)
    allowed = set(coords["StationID"].astype(str))
    metric_to_use = "NKGE"
    common = cache.common_stations(metric=metric_to_use, allowed_stations=allowed)
    print(f"Using {len(common)} common stations across runs for metric={metric_to_use}")

    # 4) Plot with the SAME stations across all figures
    figs = Path("./figs"); figs.mkdir(parents=True, exist_ok=True)

    # 5) add labels
#    labels = [
#        "MESH code v1.5.5",
#        "MESH code v1.5.6",
#        "MESH code v1.5.7"
#    ]

#    labels = [
#        "MERIT-MESH",
#        "CAMELS-SPAT-MESH",
#    ]

    labels = [
        "Calibration",
        "CAMELS-SPAT",
    ]

    # Start ploting
    plot_cumulative_distribution(cache, metric=metric_to_use, 
                                 legend_title="Model Evaluation",
                                 legend_labels=labels,
                                 output_file=figs / f"cdf_{metric_to_use.lower()}.png",
                                 common_stations=common)
    plot_broken_axis_cumulative_distribution(cache, metric="KGE",
                                             legend_title="Model Evaluation",
                                             legend_labels=labels,
                                             output_file=figs / f"cdf_{metric_to_use.lower()}_broken.png",
                                             common_stations=common, zoom_range=(-0.5, 0.8))

    plot_spatial_stations_metrics(cache, coords, metric=metric_to_use, run_index=1, run_labels=labels,
                                  output_file=figs / f"spatial_run0_{metric_to_use.lower()}.png", common_stations=common,
                                  basin_shapefile=basin_file, point_size=15)

    if len(cache.dfs) >= 2:
        plot_spatial_station_metrics_comparison(cache, coords, metric=metric_to_use,
                                                run_index1=1, run_index2=0, run_labels=labels,
                                                output_file=figs / f"spatial_cmp_{metric_to_use.lower()}.png",
                                                common_stations=common, basin_shapefile=basin_file, point_size=15,
                                                max_name_length = 20)
    else:
        print("Skipping comparison map: fewer than 2 runs loaded.")

Filter PRecord >= 10: 6192→4961
Filter HYD_STATUS == 'A': 4961→3297
Processing all model runs...
1 stations had issues.

Loading and filtering stations...
Loading 2 metric files...
Loaded 2 valid runs.
Excluded 701 stations → excluded_stations_nkge.txt
Common 172 stations → common_stations_nkge.txt
Using 172 common stations across runs for metric=NKGE
Excluded 701 stations → excluded_stations_nkge.txt
Common 172 stations → common_stations_nkge.txt
CDF saved: cdf_nkge.png
Excluded 701 stations → excluded_stations_kge.txt
Common 172 stations → common_stations_kge.txt
CDF saved: cdf_nkge_broken.png
Excluded 701 stations → excluded_stations_nkge.txt
Common 172 stations → common_stations_nkge.txt
Spatial plot saved: spatial_run0_nkge.png
Excluded 701 stations → excluded_stations_nkge.txt
Common 172 stations → common_stations_nkge.txt
Comparison plot saved: spatial_cmp_nkge.png


In [ ]:
# Load the polygon shapefiles
# Read CSV file
input_basin = r'D:\Zelalem\RUNs\MESH_CaSRv3p1\Average_GRU_Params\MESH_1p5p5_pr_forcast\metrics_MESH_CaSRv3p1_Average_GRU_Params_MESH_1p5p5_pr_forcast.csv'
csv_data = pd.read_csv(input_basin)
input_basin = r'D:\Zelalem\WSC\combined_discharge_stations_comids2.gpkg'
polygons = gpd.read_file(input_basin)
#polygons = polygons.drop(columns=['StationID', 'KGE'], errors='ignore')

In [ ]:
polygons = polygons.merge(csv_data[['StationID', 'KGE']], how='left', left_on='Obs_NM', right_on='StationID')
polygons.to_file(r'D:\Zelalem\WSC\combined_discharge_stations_comids2.gpkg', layer='points', driver="GPKG")

In [ ]:
input_basin = r'D:\Zelalem\WSC\merged-all-stations.gpkg'
gdf = gpd.read_file(input_basin)
gdf = gdf.drop_duplicates(subset="Obs_NM", keep="first")

input_basin = 'D:\\Zelalem\\WSC\\combined_discharge_stations_comids.gpkg'
polygons = gpd.read_file(input_basin)
polygons = polygons.drop(columns=['unitarea', 'uparea', 'DrainArea', 'DA_Diff', 'HYD_STATUS'])


# Save the combined station point shapefile
polygons = polygons.merge(gdf[['Obs_NM', 'DrainArea', 'DA_Diff', 'HYD_STATUS']], on='Obs_NM', how='left')
polygons = polygons.merge(csv_data[['StationID', 'KGE']], how='left', left_on='Obs_NM', right_on='StationID')

In [ ]:
# Load the polygon shapefiles
dam_in = r'D:\Zelalem\gauges-DI-updated\gauges-DI-updated.shp'
gdf1 = gpd.read_file(dam_in)

station_in = 'D:\\Zelalem\\WSC\\combined_discharge_stations_comids.gpkg'
gdf2 = gpd.read_file(station_in)

station_all = r'D:\Zelalem\WSC\merged-all-stations.gpkg'
gdf3 = gpd.read_file(station_all)

In [ ]:
# Find station names that are in gdf1 but not in gdf2
missing_stations = set(gdf1['Obs_NM']) - set(gdf2['Obs_NM'])

#print(missing_stations)
#print(f"Number of stations missing: {len(missing_stations)}")

In [ ]:
input_basin = r'D:\Zelalem\WSC\combined_discharge_stations_comids2.gpkg'
gpkg_path = r'D:\Zelalem\WSC\combined_discharge_stations_comids.shp'
polygons = gpd.read_file(input_basin)
polygons.to_file(gpkg_path)